In [22]:
import torch
import numpy as np
import scipy.special as sp

import pickle as pkl
import zlib
import base64

In [23]:
import sys
sys.path.append('/Users/aleksei/projects/code-of-kutulu-client')

In [24]:
from src.envs.agents.dqn_agent_ext import DQNAgentExt
from src.game.template import calculate_output_np

In [25]:
# checkpoint_dir = '../output/2025-05-07/23:23:14.779319/agent0'

In [26]:
# checkpoint_dir = '../output/2025-05-21/00:32:22.615450/agent0'

In [27]:
checkpoint_dir = '../output/2025-05-22/02:06:48.278418/agent1'

In [28]:
info = {
    'train': True,
    'qdn_ext': True,
    'state_type': 'closest_ext',
    'gamma': 0.5,
    'replay_size': 10000,
    'replay_start_size': 100,
    'sync_target_frames': 1000,
    'batch_size': 64,
    'action_space_n': 8,
#         'lr': 1e-5
}

In [29]:
del info['qdn_ext']

In [30]:
agent = DQNAgentExt(**info)

In [31]:
agent.model.load_state_dict(torch.load(f"{checkpoint_dir}/model.pt"))
agent.tgt_net.load_state_dict(torch.load(f"{checkpoint_dir}/model.pt"))

<All keys matched successfully>

In [32]:
data = {'entity_kind': [[1, 1, 3, 2, 0, 0, 0, 0, 0, 0]],
 'entity_features': [[[218.0, 3.0, 3.0, -3.0, 6.0, 0.0, 218.0],
   [221.0, 3.0, 1.0, -9.0, 10.0, 0.0, 221.0],
   [27.0, 0.0, 6.0, 6.0, 12.0, 0.0, 27.0],
   [2.0, -1.0, 7.0, 4.0, 11.0, 0.0, 2.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]],
 'entity_dir': [[[0.0, 6, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 12, 0.0],
   [0.0, 12, 0.0, 0.0, 0.0],
   [0.0, 15, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0]]]}

In [33]:
weights = {}
for k,v in agent.model.named_parameters():
    weights[k] = v.detach().numpy()
    print(k, v.shape)

kind_embs.weight torch.Size([13, 32])
features_linear.weight torch.Size([32, 7])
features_linear.bias torch.Size([32])
dir_linear.weight torch.Size([16, 5])
dir_linear.bias torch.Size([16])
entity_linear.weight torch.Size([16, 80])
entity_linear.bias torch.Size([16])
entity_impact.weight torch.Size([8, 80])
entity_impact.bias torch.Size([8])
out_linear.weight torch.Size([1, 16])
out_linear.bias torch.Size([1])


In [34]:
calculate_output_np(data, weights, num_classes=8)

array([[0.0054631 , 0.00507849, 0.0054677 , 0.00549429, 0.00540794,
        0.00585267, 0.00557844, 0.00463892]])

In [35]:
tensor_data = {k: torch.tensor(v) for k,v in data.items()}

In [36]:
tensor_data = {
    'entity_kind': torch.IntTensor(data['entity_kind']),
    'entity_features': torch.FloatTensor(data['entity_features']),
    'entity_dir': torch.FloatTensor(data['entity_dir']),
}

In [37]:
model_output = agent.model(tensor_data)[0].detach().cpu().numpy()

In [38]:
model_output

array([0.0054631 , 0.00507848, 0.0054677 , 0.00549429, 0.00540794,
       0.00585267, 0.00557844, 0.00463893], dtype=float32)

In [39]:
data2, data1 = zip(*weights.items())

data1 = pkl.dumps(data1)
data2 = pkl.dumps(data2)

In [40]:
with open('../src/game/template.py') as f:
    lines = f.readlines()

In [41]:
with open('../src/game/template_submit.py', 'w') as f:
    for line in lines:
        line = line.replace("b'data1data1data1'", str(base64.b64encode(zlib.compress(data1, level=9))))
        line = line.replace("b'data2data2data2'", str(base64.b64encode(zlib.compress(data2, level=9))))
        line = line.replace("mode = 'mode'", "mode = 'dqn_ext'")
        line = line.replace("USED_ACTIONS = DEFAULT_KUTULU_ACTIONS", "USED_ACTIONS = EXTENDED_KUTULU_ACTIONS")
        f.write(line)

In [42]:
!ls -lh ../src/game/template_submit.py

-rw-r--r--  1 aleksei  staff    28K 22 май 16:57 ../src/game/template_submit.py
